# EXTRACT & DATA UNDERSTANDING

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("healthcare_dataset.csv")

print("="*60)
print("DATASET INFO")
print("="*60)

print(f"Jumlah Baris  : {df.shape[0]:,}")
print(f"Jumlah Kolom  : {df.shape[1]}")

print("\nKolom Dataset:")
print(df.columns.tolist())


print("\nTipe Data")
print(df.dtypes)


print("\nMissing Value")
print(df.isnull().sum())

print("\nDuplicate Rows :", df.duplicated().sum())


print("\nPreview Dataset")
print(df.head())


df["Date of Admission"] = pd.to_datetime(
    df["Date of Admission"]
)

df["Discharge Date"] = pd.to_datetime(
    df["Discharge Date"]
)

print("\nDate Conversion Success")
print(df[[
    "Date of Admission",
    "Discharge Date"
]].head())

DATASET INFO
Jumlah Baris  : 55,500
Jumlah Kolom  : 15

Kolom Dataset:
['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results']

Tipe Data
Name                   object
Age                     int64
Gender                 object
Blood Type             object
Medical Condition      object
Date of Admission      object
Doctor                 object
Hospital               object
Insurance Provider     object
Billing Amount        float64
Room Number             int64
Admission Type         object
Discharge Date         object
Medication             object
Test Results           object
dtype: object

Missing Value
Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    

# DATA TRANSFORMATION

In [ ]:
print("="*60)
print("TRANSFORMASI DATA")
print("="*60)


text_cols = [
    "Name",
    "Gender",
    "Blood Type",
    "Medical Condition",
    "Doctor",
    "Hospital",
    "Insurance Provider",
    "Admission Type",
    "Medication",
    "Test Results"
]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

print("✓ Standardisasi text selesai")


df["Age Group"] = pd.cut(
    df["Age"],
    bins=[0,18,35,50,65,120],
    labels=[
        "0-18",
        "19-35",
        "36-50",
        "51-65",
        "65+"
    ]
)

print("✓ Age Group berhasil dibuat")


df["Length of Stay"] = (
    df["Discharge Date"]
    -
    df["Date of Admission"]
).dt.days

print("✓ Length of Stay berhasil dibuat")


df["Billing Category"] = pd.qcut(
    df["Billing Amount"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

print("✓ Billing Category berhasil dibuat")


df["Admission Year"] = (
    df["Date of Admission"]
    .dt.year
)


df["Admission Month"] = (
    df["Date of Admission"]
    .dt.month
)

df["Admission Day"] = (
    df["Date of Admission"]
    .dt.day
)

print("✓ Time Dimension berhasil dibuat")

numeric_cols = [
    "Age",
    "Billing Amount",
    "Length of Stay"
]

print("\nOUTLIER CHECK")

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_count = (
        (
            df[col] < lower
        )
        |
        (
            df[col] > upper
        )
    ).sum()

    print(
        f"{col:<20} : {outlier_count:,} outlier"
    )

print("\nDATA QUALITY CHECK")

print(
    "Missing Value :",
    df.isnull().sum().sum()
)

print(
    "Duplicate Rows :",
    df.duplicated().sum()
)

print("\nPreview Data Setelah Transformasi")

print(
    df[
        [
            "Age",
            "Age Group",
            "Billing Amount",
            "Billing Category",
            "Length of Stay",
            "Admission Year",
            "Admission Month"
        ]
    ].head()
)

print("\nJumlah Data :", len(df))

print("\n✓ Tahap Transformasi Selesai")

TRANSFORMASI DATA
✓ Standardisasi text selesai
✓ Age Group berhasil dibuat
✓ Length of Stay berhasil dibuat
✓ Billing Category berhasil dibuat
✓ Time Dimension berhasil dibuat

OUTLIER CHECK
Age                  : 0 outlier
Billing Amount       : 0 outlier
Length of Stay       : 0 outlier

DATA QUALITY CHECK
Missing Value : 0
Duplicate Rows : 534

Preview Data Setelah Transformasi
   Age Age Group  Billing Amount Billing Category  Length of Stay  \
0   30     19-35    18856.281306           Medium               2   
1   62     51-65    33643.327287             High               6   
2   76       65+    27955.096079             High              15   
3   28     19-35    37909.782410        Very High              30   
4   43     36-50    14238.317814           Medium              20   

   Admission Year  Admission Month  
0            2024                1  
1            2019                8  
2            2022                9  
3            2020               11  
4            202

# MEMBANGUN STAR SCHEMA

In [ ]:
print("="*60)
print("MEMBANGUN STAR SCHEMA")
print("="*60)


dim_admission_time = pd.DataFrame({

    "admission_date":
    pd.to_datetime(
        df["Date of Admission"]
    )

}).drop_duplicates()

dim_admission_time["tahun"] = (
    dim_admission_time["admission_date"]
    .dt.year
    .astype(str)
)

dim_admission_time["bulan"] = (
    dim_admission_time["admission_date"]
    .dt.month
    .astype(str)
)

dim_admission_time["hari"] = (
    dim_admission_time["admission_date"]
    .dt.day
    .astype(str)
)

dim_admission_time = (
    dim_admission_time
    .reset_index(drop=True)
)

dim_admission_time.insert(
    0,
    "admission_time_id",
    range(
        1,
        len(dim_admission_time)+1
    )
)

print(
    "✓ dim_admission_time :",
    dim_admission_time.shape
)



dim_discharge_time = pd.DataFrame({

    "discharge_date":
    pd.to_datetime(
        df["Discharge Date"]
    )

}).drop_duplicates()

dim_discharge_time["tahun"] = (
    dim_discharge_time["discharge_date"]
    .dt.year
    .astype(str)
)

dim_discharge_time["bulan"] = (
    dim_discharge_time["discharge_date"]
    .dt.month
    .astype(str)
)

dim_discharge_time["hari"] = (
    dim_discharge_time["discharge_date"]
    .dt.day
    .astype(str)
)

dim_discharge_time = (
    dim_discharge_time
    .reset_index(drop=True)
)

dim_discharge_time.insert(
    0,
    "discharge_time_id",
    range(
        1,
        len(dim_discharge_time)+1
    )
)

print(
    "✓ dim_discharge_time :",
    dim_discharge_time.shape
)


dim_patient = df[
    [
        "Name",
        "Gender",
        "Age",
        "Age Group",
        "Blood Type"
    ]
].copy()

dim_patient.insert(
    0,
    "patient_id",
    range(1, len(dim_patient)+1)
)

print("✓ dim_patient :", dim_patient.shape)


dim_medical = df[
    [
        "Medical Condition",
        "Medication",
        "Test Results"
    ]
].copy()

dim_medical.insert(
    0,
    "medical_id",
    range(1, len(dim_medical)+1)
)

print("✓ dim_medical :", dim_medical.shape)


dim_hospital = df[
    [
        "Hospital",
        "Doctor",
        "Insurance Provider"
    ]
].copy()

dim_hospital.insert(
    0,
    "hospital_id",
    range(1, len(dim_hospital)+1)
)

print("✓ dim_hospital :", dim_hospital.shape)

dim_admission = df[
    [
        "Admission Type",
        "Room Number"
    ]
].copy()

dim_admission.insert(
    0,
    "admission_id",
    range(1, len(dim_admission)+1)
)

print("✓ dim_admission :", dim_admission.shape)


admission_map = dict(
    zip(
        dim_admission_time["admission_date"],
        dim_admission_time["admission_time_id"]
    )
)

discharge_map = dict(
    zip(
        dim_discharge_time["discharge_date"],
        dim_discharge_time["discharge_time_id"]
    )
)

fact = df.copy()

fact["jumlah_pasien"] = 1

fact["patient_id"] = range(
    1,
    len(fact)+1
)

fact["medical_id"] = range(
    1,
    len(fact)+1
)

fact["hospital_id"] = range(
    1,
    len(fact)+1
)

fact["admission_id"] = range(
    1,
    len(fact)+1
)
fact["admission_time_id"] = (
    pd.to_datetime(
        fact["Date of Admission"]
    )
    .map(admission_map)
)

fact["discharge_time_id"] = (
    pd.to_datetime(
        fact["Discharge Date"]
    )
    .map(discharge_map)
)

fact_healthcare = fact[
    [
        "admission_time_id",
        "discharge_time_id",

        "patient_id",
        "medical_id",
        "hospital_id",
        "admission_id",

        "Billing Amount",
        "Length of Stay",

        "jumlah_pasien"
    ]
]

fact_healthcare.insert(
    0,
    "fact_id",
    range(
        1,
        len(fact_healthcare)+1
    )
)

print("✓ fact_healthcare :", fact_healthcare.shape)

print("\nNULL CHECK")

print(
    fact_healthcare
    .isnull()
    .sum()
)


print("\nJumlah Record")

print(
    "dim_admission_time :",
    len(dim_admission_time)
)

print(
    "dim_discharge_time :",
    len(dim_discharge_time)
)

print(
    "dim_patient    :",
    len(dim_patient)
)

print(
    "dim_medical    :",
    len(dim_medical)
)

print(
    "dim_hospital   :",
    len(dim_hospital)
)

print(
    "dim_admission  :",
    len(dim_admission)
)

print(
    "fact_healthcare:",
    len(fact_healthcare)
)

dim_admission_time.to_csv(
    "dim_admission_time.csv",
    index=False
)

dim_discharge_time.to_csv(
    "dim_discharge_time.csv",
    index=False
)

dim_patient.to_csv(
    "dim_patient.csv",
    index=False
)

dim_medical.to_csv(
    "dim_medical.csv",
    index=False
)

dim_hospital.to_csv(
    "dim_hospital.csv",
    index=False
)

dim_admission.to_csv(
    "dim_admission.csv",
    index=False
)

fact_healthcare.to_csv(
    "fact_healthcare.csv",
    index=False
)

print("\n✓ Semua tabel berhasil dibuat")
print("✓ CSV berhasil disimpan")

MEMBANGUN STAR SCHEMA
✓ dim_admission_time : (1827, 5)
✓ dim_discharge_time : (1856, 5)
✓ dim_patient : (55500, 6)
✓ dim_medical : (55500, 4)
✓ dim_hospital : (55500, 4)
✓ dim_admission : (55500, 3)
✓ fact_healthcare : (55500, 10)

NULL CHECK
fact_id              0
admission_time_id    0
discharge_time_id    0
patient_id           0
medical_id           0
hospital_id          0
admission_id         0
Billing Amount       0
Length of Stay       0
jumlah_pasien        0
dtype: int64

Jumlah Record
dim_admission_time : 1827
dim_discharge_time : 1856
dim_patient    : 55500
dim_medical    : 55500
dim_hospital   : 55500
dim_admission  : 55500
fact_healthcare: 55500

✓ Semua tabel berhasil dibuat
✓ CSV berhasil disimpan


# LOAD DATA WAREHOUSE KE POSTGRESQL

In [ ]:
from sqlalchemy import create_engine, text
import time

print("="*60)
print("LOAD KE POSTGRESQL")
print("="*60)


DATABASE_URL = (
    "postgresql://:divaizza@localhost:5432/Healthcare"
)

engine = create_engine(DATABASE_URL)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("✓ Koneksi PostgreSQL Berhasil")
except Exception as e:
    print("✗ Koneksi Gagal")
    print(e)


drop_sql = """
DROP TABLE IF EXISTS fact_healthcare CASCADE;
DROP TABLE IF EXISTS dim_admission CASCADE;
DROP TABLE IF EXISTS dim_hospital CASCADE;
DROP TABLE IF EXISTS dim_medical CASCADE;
DROP TABLE IF EXISTS dim_patient CASCADE;
DROP TABLE IF EXISTS dim_time CASCADE;
DROP TABLE IF EXISTS dim_admission_time CASCADE;
DROP TABLE IF EXISTS dim_discharge_time CASCADE;
"""

with engine.begin() as conn:
    conn.execute(text(drop_sql))

print("✓ Tabel lama dibersihkan")


tables = {
    "dim_admission_time":
    dim_admission_time,

    "dim_discharge_time":
    dim_discharge_time,

    "dim_patient":
    dim_patient,

    "dim_medical":
    dim_medical,

    "dim_hospital":
    dim_hospital,

    "dim_admission":
    dim_admission,

    "fact_healthcare":
    fact_healthcare
}

print("\nUpload Data...")

for nama, tabel in tables.items():

    start = time.time()

    tabel.to_sql(
        nama,
        engine,
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=5000
    )

    durasi = time.time() - start

    print(
        f"✓ {nama:<20} "
        f"{len(tabel):>8,} rows "
        f"({durasi:.2f}s)"
    )

print("\n✓ Semua tabel berhasil diupload")

pk_sql = """

ALTER TABLE dim_admission_time
ADD PRIMARY KEY (admission_time_id);

ALTER TABLE dim_discharge_time
ADD PRIMARY KEY (discharge_time_id);

ALTER TABLE dim_patient
ADD PRIMARY KEY (patient_id);

ALTER TABLE dim_medical
ADD PRIMARY KEY (medical_id);

ALTER TABLE dim_hospital
ADD PRIMARY KEY (hospital_id);

ALTER TABLE dim_admission
ADD PRIMARY KEY (admission_id);

ALTER TABLE fact_healthcare
ADD PRIMARY KEY (fact_id);

"""

with engine.begin() as conn:
    conn.execute(text(pk_sql))

print("✓ Primary Key berhasil dibuat")


fk_sql = """
ALTER TABLE fact_healthcare
ADD CONSTRAINT fk_admission_time
FOREIGN KEY (admission_time_id)
REFERENCES dim_admission_time(admission_time_id);

ALTER TABLE fact_healthcare
ADD CONSTRAINT fk_discharge_time
FOREIGN KEY (discharge_time_id)
REFERENCES dim_discharge_time(discharge_time_id);

ALTER TABLE fact_healthcare
ADD CONSTRAINT fk_patient
FOREIGN KEY (patient_id)
REFERENCES dim_patient(patient_id);

ALTER TABLE fact_healthcare
ADD CONSTRAINT fk_medical
FOREIGN KEY (medical_id)
REFERENCES dim_medical(medical_id);

ALTER TABLE fact_healthcare
ADD CONSTRAINT fk_hospital
FOREIGN KEY (hospital_id)
REFERENCES dim_hospital(hospital_id);

ALTER TABLE fact_healthcare
ADD CONSTRAINT fk_admission
FOREIGN KEY (admission_id)
REFERENCES dim_admission(admission_id);

"""

with engine.begin() as conn:
    conn.execute(text(fk_sql))

print("✓ Foreign Key berhasil dibuat")


print("\nVerifikasi PostgreSQL")

with engine.connect() as conn:

    daftar_tabel = [
        "dim_patient",
        "dim_medical",
        "dim_hospital",
        "dim_admission",
        "fact_healthcare",
        "dim_admission_time",
        "dim_discharge_time",
    ]

    for tabel in daftar_tabel:

        total = conn.execute(
            text(
                f"SELECT COUNT(*) FROM {tabel}"
            )
        ).scalar()

        print(
            f"{tabel:<20}: "
            f"{total:,} rows"
        )

print("\n✓ Star Schema berhasil dibuat di PostgreSQL")

print("\nDATABASE READY FOR OLAP")

LOAD KE POSTGRESQL
✓ Koneksi PostgreSQL Berhasil
✓ Tabel lama dibersihkan

Upload Data...
✓ dim_admission_time      1,827 rows (0.09s)
✓ dim_discharge_time      1,856 rows (0.14s)
✓ dim_patient            55,500 rows (6.43s)
✓ dim_medical            55,500 rows (5.09s)
✓ dim_hospital           55,500 rows (3.18s)
✓ dim_admission          55,500 rows (4.62s)
✓ fact_healthcare        55,500 rows (8.83s)

✓ Semua tabel berhasil diupload
✓ Primary Key berhasil dibuat
✓ Foreign Key berhasil dibuat

Verifikasi PostgreSQL
dim_patient         : 55,500 rows
dim_medical         : 55,500 rows
dim_hospital        : 55,500 rows
dim_admission       : 55,500 rows
fact_healthcare     : 55,500 rows
dim_admission_time  : 1,827 rows
dim_discharge_time  : 1,856 rows

✓ Star Schema berhasil dibuat di PostgreSQL

DATABASE READY FOR OLAP


# POSTGRESQL OPTIMIZATION
# INDEX + MATERIALIZED VIEW + BENCHMARK

In [ ]:
from sqlalchemy import text
import pandas as pd
import time

print("="*60)
print("POSTGRESQL OPTIMIZATION")
print("="*60)

index_sql = [

    """
    CREATE INDEX IF NOT EXISTS idx_fact_admission_time
    ON fact_healthcare(admission_time_id);
    """

    """
    CREATE INDEX IF NOT EXISTS idx_fact_discharge_time
    ON fact_healthcare(discharge_time_id);
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_fact_patient
    ON fact_healthcare(patient_id)
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_fact_medical
    ON fact_healthcare(medical_id)
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_fact_hospital
    ON fact_healthcare(hospital_id)
    """,

    """
    CREATE INDEX IF NOT EXISTS idx_fact_admission
    ON fact_healthcare(admission_id)
    """

]

with engine.begin() as conn:

    for q in index_sql:
        conn.execute(text(q))

print("✓ Index berhasil dibuat")


with engine.begin() as conn:

    conn.execute(
        text(
            """
            DROP MATERIALIZED VIEW IF EXISTS
            mv_healthcare_summary
            """
        )
    )

print("✓ Materialized View lama dihapus")


mv_sql = """

CREATE MATERIALIZED VIEW mv_healthcare_summary AS

SELECT

    t.tahun                           AS tahun,
    t.bulan                           AS bulan,

    p."Gender"                        AS gender,

    m."Medical Condition"             AS kondisi,

    COUNT(*)                          AS total_pasien,

    ROUND(
        AVG(f."Billing Amount")::numeric,
        2
    ) AS rata_billing,

    ROUND(
        AVG(f."Length of Stay")::numeric,
        2
    ) AS rata_los

FROM fact_healthcare f

JOIN dim_admission_time t
ON f.admission_time_id = t.admission_time_id

JOIN dim_patient p
ON f.patient_id = p.patient_id

JOIN dim_medical m
ON f.medical_id = m.medical_id

GROUP BY

    t.tahun,
    t.bulan,
    p."Gender",
    m."Medical Condition"

"""

with engine.begin() as conn:

    conn.execute(text(mv_sql))

print("✓ Materialized View berhasil dibuat")

mv_index_sql = """

CREATE INDEX IF NOT EXISTS
idx_mv_healthcare

ON mv_healthcare_summary(
    tahun,
    gender
)

"""

with engine.begin() as conn:

    conn.execute(text(mv_index_sql))

print("✓ Index MV berhasil dibuat")

print("\nPreview Materialized View")

preview = pd.read_sql(
    """
    SELECT *
    FROM mv_healthcare_summary
    LIMIT 10
    """,
    engine
)

print(preview)


query_join = """

SELECT

    t.tahun,
    t.bulan,

    p."Gender",

    m."Medical Condition",

    COUNT(*) total_pasien,

    AVG(f."Billing Amount") avg_bill,

    AVG(f."Length of Stay") avg_los

FROM fact_healthcare f

JOIN dim_admission_time t
ON f.admission_time_id = t.admission_time_id

JOIN dim_patient p
ON f.patient_id = p.patient_id

JOIN dim_medical m
ON f.medical_id = m.medical_id

GROUP BY

    t.tahun,
    t.bulan,
    p."Gender",
    m."Medical Condition"

"""


query_mv = """

SELECT *

FROM mv_healthcare_summary

"""


hasil = {}

with engine.connect() as conn:

    for label, query in [

        ("JOIN Langsung", query_join),

        ("Materialized View", query_mv)

    ]:

        waktu = []

        for i in range(5):

            start = time.time()

            conn.execute(
                text(query)
            ).fetchall()

            waktu.append(
                time.time() - start
            )

        hasil[label] = (
            sum(waktu)
            /
            len(waktu)
        ) * 1000

print("\n")

print("="*50)
print(f"{'METHOD':<25} {'AVG TIME (ms)':>15}")
print("="*50)

for k,v in hasil.items():

    print(
        f"{k:<25} {v:>10.2f}"
    )

print("="*50)

speedup = (
    hasil["JOIN Langsung"]
    /
    hasil["Materialized View"]
)

print(
    f"\n✓ MV lebih cepat "
    f"{speedup:.2f}x"
)

print("\nEXPLAIN ANALYZE")

plan = pd.read_sql(

    """
    EXPLAIN ANALYZE
    SELECT *
    FROM mv_healthcare_summary
    """,

    engine

)

print(plan)

print("\n✓ Tahap Optimasi PostgreSQL selesai")

POSTGRESQL OPTIMIZATION
✓ Index berhasil dibuat
✓ Materialized View lama dihapus
✓ Materialized View berhasil dibuat
✓ Index MV berhasil dibuat

Preview Materialized View
   tahun  gender       kondisi  total_pasien  rata_billing  rata_los
0   2024  Female      Diabetes           319      23487.64     15.19
1   2023    Male        Cancer           949      25006.25     15.97
2   2021    Male  Hypertension           947      25736.10     15.68
3   2021  Female       Obesity           945      26106.03     15.50
4   2023  Female       Obesity           921      25935.92     15.65
5   2019    Male  Hypertension           618      25248.10     15.70
6   2022  Female        Cancer           941      25170.73     15.21
7   2019  Female        Asthma           628      25926.74     15.91
8   2023  Female  Hypertension           951      25281.87     14.94
9   2023    Male       Obesity           940      26228.80     15.31


METHOD                      AVG TIME (ms)
JOIN Langsung             

# OLAP CUBE (ATOTI)

In [ ]:
import atoti as tt

print("="*60)
print("MEMBANGUN OLAP CUBE")
print("="*60)

session = tt.Session.start()

print("✓ Atoti Started")

fact_store = session.read_pandas(
    fact_healthcare,
    table_name="fact_healthcare",
    keys=["fact_id"]
)

admission_time_store = session.read_pandas(
    dim_admission_time,
    table_name="dim_admission_time",
    keys=["admission_time_id"]
)

discharge_time_store = session.read_pandas(
    dim_discharge_time,
    table_name="dim_discharge_time",
    keys=["discharge_time_id"]
)

patient_store = session.read_pandas(
    dim_patient,
    table_name="dim_patient",
    keys=["patient_id"]
)

medical_store = session.read_pandas(
    dim_medical,
    table_name="dim_medical",
    keys=["medical_id"]
)

hospital_store = session.read_pandas(
    dim_hospital,
    table_name="dim_hospital",
    keys=["hospital_id"]
)

admission_store = session.read_pandas(
    dim_admission,
    table_name="dim_admission",
    keys=["admission_id"]
)

print("✓ Semua store berhasil dibuat")

fact_store.join(
    admission_time_store,
    fact_store["admission_time_id"]
    ==
    admission_time_store["admission_time_id"]
)

fact_store.join(
    discharge_time_store,
    fact_store["discharge_time_id"]
    ==
    discharge_time_store["discharge_time_id"]
)

fact_store.join(
    patient_store,
    fact_store["patient_id"] == patient_store["patient_id"]
)

fact_store.join(
    medical_store,
    fact_store["medical_id"] == medical_store["medical_id"]
)

fact_store.join(
    hospital_store,
    fact_store["hospital_id"] == hospital_store["hospital_id"]
)

fact_store.join(
    admission_store,
    fact_store["admission_id"] == admission_store["admission_id"]
)

print("✓ Join Star Schema berhasil")

cube = session.create_cube(
    fact_store,
    name="HealthcareCube"
)

print("✓ Cube berhasil dibuat")

print("\nLEVEL YANG TERDETEKSI:")

for lvl in cube.levels:
    print(lvl)


lvl = cube.levels


cube.hierarchies["Admission Time"] = {

    "Tahun":
    lvl[
        (
            "dim_admission_time",
            "tahun",
            "tahun"
        )
    ],

    "Bulan":
    lvl[
        (
            "dim_admission_time",
            "bulan",
            "bulan"
        )
    ],

    "Hari":
    lvl[
        (
            "dim_admission_time",
            "hari",
            "hari"
        )
    ]
}

cube.hierarchies["Discharge Time"] = {

    "Tahun":
    lvl[
        (
            "dim_discharge_time",
            "tahun",
            "tahun"
        )
    ],

    "Bulan":
    lvl[
        (
            "dim_discharge_time",
            "bulan",
            "bulan"
        )
    ],

    "Hari":
    lvl[
        (
            "dim_discharge_time",
            "hari",
            "hari"
        )
    ]
}


cube.hierarchies["Pasien"] = {

    "Gender":
    lvl[("dim_patient","Gender","Gender")],

    "Age Group":
    lvl[("dim_patient","Age Group","Age Group")]
}


cube.hierarchies["Medical"] = {

    "Condition":
    lvl[("dim_medical","Medical Condition","Medical Condition")],

    "Test Result":
    lvl[("dim_medical","Test Results","Test Results")]
}


cube.hierarchies["Hospital"] = {

    "Hospital":
    lvl[("dim_hospital","Hospital","Hospital")],

    "Insurance":
    lvl[("dim_hospital","Insurance Provider","Insurance Provider")]
}


cube.hierarchies["Admission"] = {

    "Type":
    lvl[("dim_admission","Admission Type","Admission Type")]
}

print("✓ Hierarchy berhasil dibuat")

meas = cube.measures


meas["Total Pasien"] = tt.agg.sum(
    fact_store["jumlah_pasien"]
)


meas["Total Billing"] = tt.agg.sum(
    fact_store["Billing Amount"]
)


meas["Avg Billing"] = tt.agg.mean(
    fact_store["Billing Amount"]
)

meas["Avg Billing Per Patient"] = (

    meas["Total Billing"]

    /

    meas["Total Pasien"]

)

meas["Avg Length Of Stay"] = tt.agg.mean(
    fact_store["Length of Stay"]
)
meas["Max Length Of Stay"] = tt.agg.max(
    fact_store["Length of Stay"]
)
meas["Min Length Of Stay"] = tt.agg.min(
    fact_store["Length of Stay"]
)

print("✓ Measures berhasil dibuat")

print("\n=== HIERARCHIES ===")
print(list(cube.hierarchies))

print("\n=== MEASURES ===")
print(list(cube.measures))

print("\nDashboard URL:")
print(session.url)

print("\n✓ OLAP Cube Selesai Dibuat")

Welcome to Atoti 0.9.13!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.
MEMBANGUN OLAP CUBE
✓ Atoti Started
✓ Semua store berhasil dibuat
✓ Join Star Schema berhasil
✓ Cube berhasil dibuat

LEVEL YANG TERDETEKSI:
('fact_healthcare', 'fact_id', 'fact_id')
('dim_medical', 'Test Results', 'Test Results')
('dim_medical', 'Medical Condition', 'Medical Condition')
('dim_medical', 'Medication', 'Medication')
('dim_patient', 'Age Group', 'Age Grou